In [1]:
# ============================================================
# 1. 라이브러리 불러오기
# ============================================================

import pandas as pd
import numpy as np

from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format','{:.4f}'.format)

In [2]:
# ============================================================
# 2. 데이터 불러오기
# ============================================================

train = pd.read_csv("../../data/processed/train_fe.csv")

In [3]:
# ============================================================
# 3. 7번 파일 결과 불러오기
# - 변수 제거 검증 결과에서 가장 성능이 좋았던 feature set을 사용한다.
# ============================================================

feature_set_summary_df = pd.read_csv("../../output/tables/feature_set_cv_summary.csv")

best_feature_set = feature_set_summary_df.iloc[0]['feature_set']

print("[7번 파일 기준 최종 선택 변수 선택]")
print(best_feature_set)

[7번 파일 기준 최종 선택 변수 선택]
derived_features_only


In [4]:
# ============================================================
# 4. 변수 세트별 제거 컬럼 정의
# - 7번 파일과 동일하게 맞춰야 한다. 
# ============================================================

drop_cols_by_feature_set = {
    'derived_features_only': [
        'PassengerId',
        'Name',
        'Ticket',
        'Cabin'
    ],

    'without_title_info': [
        'PassengerId',
        'Name',
        'Ticket',
        'Cabin',
        'Title'
    ],

    'without_ticket_info': [
        'PassengerId',
        'Name',
        'Ticket',
        'Cabin',
        'TicketPrefix'
    ],

    'without_cabin_info': [
        'PassengerId',
        'Name',
        'Ticket',
        'Cabin',
        'CabinDeck'
    ],

    'without_bands': [
        'PassengerId',
        'Name',
        'Ticket',
        'Cabin',
        'AgeBand',
        'FareBand'
    ],
 
    'Without_family_size': [
        'PassengerId',
        'Name',
        'Ticket',
        'Cabin',
        'FamilySize'
    ],

    'Without_is_alone': [
        'PassengerId',
        'Name',
        'Ticket',
        'Cabin',
        'IsAlone'
    ],

    'Without_family_info': [
        'PassengerId',
        'Name',
        'Ticket',
        'Cabin',
        'FamilySize',
        'IsAlone'
    ],

    'basic_features_only': [
        'PassengerId',
        'Name',
        'Ticket',
        'Cabin',
        'Title',
        'FamilySize',
        'IsAlone',
        'AgeBand',
        'Fare_log',
        'FareBand',
        'CabinDeck',
        'TicketPrefix'
    ]
}

In [5]:
# ============================================================
# 5. 모델링용 데이터 준비
# ============================================================

drop_cols = drop_cols_by_feature_set[best_feature_set]

train_model = train.drop(columns=drop_cols)

X = train_model.drop(columns=['Survived'])
y = train_model['Survived']

X = pd.get_dummies(X, drop_first=True)

print("\n[모델 비교 입력 데이터 확인]")
print("X shape:", X.shape)
print("y shape:", y.shape)

print("\n[사용 피처 목록]")
print(X.columns.tolist())


[모델 비교 입력 데이터 확인]
X shape: (891, 37)
y shape: (891,)

[사용 피처 목록]
['Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'FamilySize', 'IsAlone', 'Fare_log', 'Sex_male', 'Embarked_Q', 'Embarked_S', 'Title_Miss', 'Title_Mr', 'Title_Mrs', 'Title_Rare', 'AgeBand_Child', 'AgeBand_Senior', 'AgeBand_Teen', 'AgeBand_YoungAdult', 'FareBand_Low', 'FareBand_MidHigh', 'FareBand_MidLow', 'CabinDeck_B', 'CabinDeck_C', 'CabinDeck_D', 'CabinDeck_E', 'CabinDeck_F', 'CabinDeck_G', 'CabinDeck_M', 'CabinDeck_T', 'TicketPrefix_CA', 'TicketPrefix_NONE', 'TicketPrefix_PC', 'TicketPrefix_Rare', 'TicketPrefix_SOTONOQ', 'TicketPrefix_STONO', 'TicketPrefix_WC']


In [6]:
# ============================================================
# 6. StrarifiedKFold 설정
# ============================================================

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [7]:
# ============================================================
# 7. 비교 모델 정의
# - 선형, 비선형, 앙상블, 고급 앙상블 
#   모델 복잡도와 학습 방식이 다른 것들을 비교한다
# ============================================================

models = {
    'LogisticRegression' : LogisticRegression(
        max_iter=1000,
        random_state=42
    ),
    'DecisionTree' : DecisionTreeClassifier(
        max_depth=5,
        random_state=42
    ),
    'RandomForest' : RandomForestClassifier(
        n_estimators=300,
        max_depth=5,
        random_state=42
    ),
    'GradientBoosting': GradientBoostingClassifier(
        random_state=42
    )   
}

In [8]:
# ============================================================
# 8. 모델별 교차검증 수행
# ============================================================

fold_results = []

for model_name, model in models.items():

    print("=" * 60)
    print(f"{model_name} 교차검증 시작")
    print("=" * 60)

    for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y), 1) :

        X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
        y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]

        fold_model = clone(model)

        fold_model.fit(X_train, y_train)

        valid_pred = fold_model.predict(X_valid)

        acc = accuracy_score(y_valid, valid_pred)
        prec = precision_score(y_valid, valid_pred, zero_division=0)
        rec = recall_score(y_valid, valid_pred, zero_division=0)
        f1 = f1_score(y_valid, valid_pred, zero_division=0)

        fold_results.append({
            'feature_set' : best_feature_set,
            'model': model_name,
            'fold' : fold,
            'accuracy' : acc,
            'precision' : prec,
            'recall' : rec,
            'f1_score' : f1
        })

        print(f"""
        [Fold {fold}]
        acc={acc:.4f}
        precision={prec:.4f}
        recall={rec:.4f}
        f1={f1:.4f}
            """
        )

LogisticRegression 교차검증 시작

        [Fold 1]
        acc=0.8380
        precision=0.7941
        recall=0.7826
        f1=0.7883
            

        [Fold 2]
        acc=0.8146
        precision=0.7778
        recall=0.7206
        f1=0.7481
            

        [Fold 3]
        acc=0.7921
        precision=0.7627
        recall=0.6618
        f1=0.7087
            

        [Fold 4]
        acc=0.8202
        precision=0.7812
        recall=0.7353
        f1=0.7576
            

        [Fold 5]
        acc=0.8146
        precision=0.7500
        recall=0.7826
        f1=0.7660
            
DecisionTree 교차검증 시작

        [Fold 1]
        acc=0.8436
        precision=0.7971
        recall=0.7971
        f1=0.7971
            

        [Fold 2]
        acc=0.8315
        precision=0.8276
        recall=0.7059
        f1=0.7619
            

        [Fold 3]
        acc=0.7697
        precision=0.6901
        recall=0.7206
        f1=0.7050
            

        [Fold 4]
        acc=0.

In [9]:
# ============================================================
# 9. Fold별 결과 정리
# ============================================================

model_fold_results_df = pd.DataFrame(fold_results)

print("\n[모델별 Fold 성능]")
display(model_fold_results_df)


[모델별 Fold 성능]


,feature_set,model,fold,accuracy,precision,recall,f1_score
0,derived_features_only,LogisticRegression,1,0.8380,0.7941,0.7826,0.7883
1,derived_features_only,LogisticRegression,2,0.8146,0.7778,0.7206,0.7481
2,derived_features_only,LogisticRegression,3,0.7921,0.7627,0.6618,0.7087
3,derived_features_only,LogisticRegression,4,0.8202,0.7812,0.7353,0.7576
4,derived_features_only,LogisticRegression,5,0.8146,0.7500,0.7826,0.7660
5,derived_features_only,DecisionTree,1,0.8436,0.7971,0.7971,0.7971
6,derived_features_only,DecisionTree,2,0.8315,0.8276,0.7059,0.7619
7,derived_features_only,DecisionTree,3,0.7697,0.6901,0.7206,0.7050
8,derived_features_only,DecisionTree,4,0.8202,0.8333,0.6618,0.7377
9,derived_features_only,DecisionTree,5,0.8371,0.7857,0.7971,0.7914


In [10]:
# ============================================================
# 10. 모델별 평균/표준편차 성능 비교
# ============================================================

model_comparison_df = (
    model_fold_results_df
    .groupby('model')
    .agg(
        accuracy_mean=('accuracy','mean'),
        accuracy_std=('accuracy','std'),
        precision_mean=('precision','mean'),
        precision_std=('precision','std'),
        recall_mean=('recall','mean'),
        recall_std=('recall','std'),
        f1_mean=('f1_score','mean'),
        f1_std=('f1_score','std')
    )
    .reset_index()
    .sort_values(by='f1_mean', ascending=False)
)

print("\n[모델 평균 성능 비교]")
display(model_comparison_df)


[모델 평균 성능 비교]


,model,accuracy_mean,accuracy_std,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std
3,RandomForest,0.8339,0.0164,0.8154,0.0223,0.7337,0.0494,0.7715,0.0292
1,GradientBoosting,0.8339,0.0182,0.8145,0.0115,0.7337,0.0504,0.7715,0.0330
0,DecisionTree,0.8204,0.0296,0.7868,0.0576,0.7365,0.0594,0.7586,0.0383
2,LogisticRegression,0.8159,0.0164,0.7732,0.0171,0.7366,0.0502,0.7537,0.0293


In [11]:
# ============================================================
# 11. 최종 모델 후보 선택
# ============================================================

best_model_name = model_comparison_df.iloc[0]['model']
best_f1 = model_comparison_df.iloc[0]['f1_mean']

print(f"""
[최종 모델 후보]\n
Best model: {best_model_name}
Selected Feature Set : {best_feature_set}
Mean F1 Score : {best_f1:.4f}
"""
)


[최종 모델 후보]

Best model: RandomForest
Selected Feature Set : derived_features_only
Mean F1 Score : 0.7715



In [12]:
# ============================================================
# 12. 결과 저장
# ============================================================

model_fold_results_df.to_csv("../../output/tables/model_comparison_fold_results.csv", index=False)

model_comparison_df.to_csv("../../output/tables/model_comparison_results.csv", index=False)

print("모델 비교 결과 저장 완료")

모델 비교 결과 저장 완료
